Author: Krish with additions from Ella and Liv

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

# Reading data

In [3]:
data_path = "/Users/ellammarks/Desktop/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

In [4]:
adhoc_data_path = "/Users/ellammarks/Desktop/Adhoc datasets"

In [5]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [6]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [7]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [8]:
df_main.shape

(3328626, 8)

In [9]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [10]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [11]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [12]:
df_main.reset_index(inplace = True, drop = True)

In [13]:
df_main['Date'] = df_main['Activity Start Timestamp'].dt.date

# Data prep for dashboard
## Data for the Menu Traffic and Hourly trends tabs of the dashboard (link below)
https://app.powerbi.com/groups/me/reports/01ca1c28-e56b-42fe-89c9-4cd53882a4c7/c1da62005af463baaa15?experience=power-bi

In [56]:
# Next activity and the one after that (per session), given rows are already sorted
def dash_data(ep_name, activity_name):
    familymenu_rows = df_main.loc[df_main['EP Name'] == ep_name,:]
    familymenu_data = df_main.loc[df_main['Contact Session ID'].isin(familymenu_rows['Contact Session ID']),:]
    familymenu_activity = familymenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    familymenu_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df2 = familymenu_activity.copy()
    
    g = df2.groupby("Contact Session ID")
    df2["Second Activity"]      = g["Activity Name"].shift(-1)
    df2["Third Activity"]  = g["Activity Name"].shift(-2)
    df2["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df2["Fifth Activity"]  = g["Activity Name"].shift(-4)
    
    out = (
        df2.loc[df2["Activity Name"].eq(activity_name),
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    # select the 3rd and 4th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 6]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                  .str.replace("FamilySP", "Family", regex=False)
                  .str.replace("EducationSP", "Education", regex=False)
                  .str.replace("ConsumerSP", "Consumer", regex=False)
                  .str.replace("BenefitsSP", "Consumer", regex=False)
                  .str.replace("EmploymentSP", "Employment", regex=False)
                  .str.replace("ImmigrationSP", "Immigration", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
        )
    return out

In [57]:
out_family = dash_data('Legal Family Menu Telephony EP', 'FamilyMenu')
out_housing = dash_data('Legal Housing Menu Telephony EP', 'HousingMenu')
out_benefits = dash_data('Legal Benefits Menu Telephony EP', 'BenefitsMenu')
out_hiv = dash_data('Legal HIV Menu Telephony EP', 'HIVMenu')
out_imm = dash_data('Legal Immigration Menu Telephony EP', 'ImmigrationMenu')
out_emp = dash_data('Legal Employment Menu Telephony EP', 'EmploymentMenu')

out_all = pd.concat([out_family, out_housing, out_benefits, out_hiv, out_imm, out_emp]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'family_dash.csv', index = False)

In [14]:
# Next activity and the one after that (per session), given rows are already sorted
def dash_data2(ep_name, activity_name):
    prelegalseniorsmenu_rows = df_main.loc[df_main['EP Name'] == 'Pre-Legal Menu Seniors Menu Telephony EP',:]
    prelegalseniorsmenu_data = df_main.loc[df_main['Contact Session ID'].isin(prelegalseniorsmenu_rows['Contact Session ID']),:]
    prelegalseniorsmenu_activity = prelegalseniorsmenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    prelegalseniorsmenu_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df3 = prelegalseniorsmenu_activity.copy()
    
    g = df3.groupby("Contact Session ID")
    df3["Second Activity"]      = g["Activity Name"].shift(-1)
    df3["Third Activity"]  = g["Activity Name"].shift(-2)
    df3["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df3["Fifth Activity"]  = g["Activity Name"].shift(-4)
    
    out = (
        df3.loc[df3["Activity Name"].eq(activity_name),
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    # select the 3rd and 4th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 6]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                  .str.replace("FamilySP", "Family", regex=False)
                  .str.replace("EducationSP", "Education", regex=False)
                  .str.replace("ConsumerSP", "Consumer", regex=False)
                  .str.replace("BenefitsSP", "Consumer", regex=False)
                  .str.replace("EmploymentSP", "Employment", regex=False)
                  .str.replace("ImmigrationSP", "Immigration", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
        )
    return out

In [17]:
out_seniors = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SeniorsMenu')
out_snrsconf = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SeniorsConfirmationMenu')
out_subscity = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SuburbsOrCityMenu')
out_subsnrs = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SuburbanSeniorsMenu')
out_seniorsadapt = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SeniorsADAPTMenu')
out_snrnotcook = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SeniorNotCookCoMenu')
out_thxbye = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'ThankYouGoodbye')
out_disconnect1 = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'DisconnectContact1')
out_pretenant = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'PreTenantMenu')
out_persinjury = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'OtherLegalPersonalInjury')
out_legal = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'LegalMenu')

out_all = pd.concat([out_seniors, out_snrsconf, out_subscity, out_subsnrs, out_seniorsadapt, out_snrnotcook, out_thxbye, out_disconnect1, out_pretenant, out_persinjury, out_legal]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'prelegalseniors_dash.csv', index = False)